# Actividad 5 - Entrenamiento y registro de experimentos
Dataset Titanic (clasificacion). Comparacion de **Regresion Logistica** vs **Random Forest**, ajuste de hiperparametros con GridSearch + validacion cruzada y registro en **MLflow**.

En Google Colab ejecuta primero la celda de instalacion.

In [ ]:
# Celda 1: instalacion (solo en Colab)
!pip -q install mlflow scikit-learn seaborn scipy

In [ ]:
# Celda 2: librerias
import warnings, numpy as np, pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
import mlflow, mlflow.sklearn
warnings.filterwarnings('ignore')
SEED = 42

In [ ]:
# Celda 3: carga y limpieza (mismas reglas que datos_prep.py)
crudo = sns.load_dataset('titanic')
df = crudo.drop(columns=['alive','adult_male','who','class','embark_town','deck','alone'])
df = df[(df['fare'].isna()) | (df['fare'] >= 0)]
df = df[(df['age'].isna()) | (df['age'] > 0)]
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['sex'] = df['sex'].map({'male':0,'female':1}).astype(int)
df = pd.get_dummies(df, columns=['embarked'], prefix='emb')
df[df.select_dtypes('bool').columns] = df.select_dtypes('bool').astype(int)
print(df.shape, 'nulos:', int(df.isna().sum().sum()))
df.head()

In [ ]:
# Celda 4: visualizaciones (patrones y anomalias)
fig, ax = plt.subplots(1,3, figsize=(15,4))
sns.barplot(data=df, x='pclass', y='survived', ax=ax[0]); ax[0].set_title('Supervivencia por clase')
sns.barplot(data=df, x='sex', y='survived', ax=ax[1]); ax[1].set_title('Supervivencia por sexo')
sns.boxplot(data=df, x='survived', y='fare', ax=ax[2]); ax[2].set_title('Tarifa (anomalias)')
plt.tight_layout(); plt.show()

In [ ]:
# Celda 5: split + configuracion MLflow
X = df.drop(columns=['survived']); y = df['survived']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
mlflow.set_tracking_uri('http://127.0.0.1:5000')  # en Colab usa './mlruns'
mlflow.set_experiment('Actividad5_Titanic')

In [ ]:
# Celda 6: entrenamiento, ajuste y registro en MLflow
configs = {
 'RegresionLogistica': (Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(max_iter=1000,random_state=SEED))]),
                        {'clf__C':[0.01,0.1,1,10]}),
 'RandomForest': (Pipeline([('clf',RandomForestClassifier(random_state=SEED))]),
                  {'clf__n_estimators':[100,300],'clf__max_depth':[None,5,10],'clf__min_samples_split':[2,5]}),
}
modelos, tabla, cvsc = {}, {}, {}
for nombre,(pipe,grid) in configs.items():
    with mlflow.start_run(run_name=nombre):
        gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1', n_jobs=-1).fit(X_tr,y_tr)
        m = gs.best_estimator_
        sc = cross_val_score(m, X_tr, y_tr, cv=cv, scoring='f1'); cvsc[nombre]=sc
        pred=m.predict(X_te); proba=m.predict_proba(X_te)[:,1]
        met={'accuracy':accuracy_score(y_te,pred),'precision':precision_score(y_te,pred),
             'recall':recall_score(y_te,pred),'f1':f1_score(y_te,pred),'roc_auc':roc_auc_score(y_te,proba)}
        modelos[nombre]=m; tabla[nombre]=met
        mlflow.log_params(gs.best_params_)
        for k,v in met.items(): mlflow.log_metric(f'test_{k}',v)
        mlflow.log_metric('cv_f1_mean', sc.mean()); mlflow.log_metric('cv_f1_std', sc.std())
        mlflow.sklearn.log_model(m, name='modelo')
        print(nombre, gs.best_params_, {k:round(v,3) for k,v in met.items()})
pd.DataFrame(tabla).T.round(4)

In [ ]:
# Celda 7: curva ROC comparativa
for n,m in modelos.items():
    pr=m.predict_proba(X_te)[:,1]; fpr,tpr,_=roc_curve(y_te,pr)
    plt.plot(fpr,tpr,label=f'{n} (AUC={roc_auc_score(y_te,pr):.3f})')
plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.title('Curva ROC'); plt.show()

In [ ]:
# Celda 8: comparacion estadistica (t pareada y Wilcoxon sobre los folds)
a,b = list(cvsc.values()); n=list(cvsc.keys())
t,tp = stats.ttest_rel(a,b); w,wp = stats.wilcoxon(a,b)
print(f'F1 CV {n[0]}: {a.mean():.4f}+/-{a.std():.4f}')
print(f'F1 CV {n[1]}: {b.mean():.4f}+/-{b.std():.4f}')
print(f't pareada p={tp:.4f} | Wilcoxon p={wp:.4f}')
print('Significativa (a=0.05):', 'SI' if tp<0.05 else 'NO')